# Hidden Markov Models from Scratch

**Goal:** Implement the core HMM algorithms — Forward, Backward, Viterbi, Baum-Welch — from first principles using NumPy.

## Intuition

An HMM models a system that transitions through **hidden states** over time, where each hidden state emits an **observable symbol** according to some probability distribution. We only see the observations, never the states directly.

**Markov property:** The next state depends *only* on the current state, not the full history: $P(z_t | z_{t-1}, z_{t-2}, \ldots) = P(z_t | z_{t-1})$.

### HMM Parameters ($\lambda = (\pi, A, B)$)

| Symbol | Name | Shape | Meaning |
|--------|------|-------|---------|
| $\pi$ | Initial distribution | $(N,)$ | $\pi_i = P(z_1 = i)$ |
| $A$ | Transition matrix | $(N, N)$ | $a_{ij} = P(z_{t+1}=j \mid z_t=i)$ |
| $B$ | Emission matrix | $(N, M)$ | $b_i(k) = P(x_t=k \mid z_t=i)$ |

where $N$ = number of hidden states, $M$ = number of observation symbols.

### The Three Fundamental HMM Problems

| # | Problem | Algorithm | Question |
|---|---------|-----------|----------|
| 1 | Evaluation | **Forward** (or Backward) | Given model $\lambda$ and observations $O$, what is $P(O \mid \lambda)$? |
| 2 | Decoding | **Viterbi** | Given model $\lambda$ and observations $O$, what is the most likely state sequence? |
| 3 | Learning | **Baum-Welch** (EM) | Given observations $O$, learn $\lambda$ that maximizes $P(O \mid \lambda)$. |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)
np.set_printoptions(precision=4, suppress=True)

## 1. Define a Weather / Activity HMM

Classic toy example:
- **Hidden states:** Sunny (0), Rainy (1)
- **Observations:** Walk (0), Shop (1), Clean (2)

In [ ]:
# --- HMM definition ---
states = ['Sunny', 'Rainy']
observations_vocab = ['Walk', 'Shop', 'Clean']
N = len(states)       # 2 hidden states
M = len(observations_vocab)  # 3 observation symbols

# Initial state distribution  pi[i] = P(z_1 = i)
pi = np.array([0.6, 0.4])

# Transition matrix  A[i,j] = P(z_{t+1}=j | z_t=i)
A = np.array([
    [0.7, 0.3],   # Sunny -> Sunny=0.7, Rainy=0.3
    [0.4, 0.6],   # Rainy -> Sunny=0.4, Rainy=0.6
])

# Emission matrix  B[i,k] = P(x_t=k | z_t=i)
B = np.array([
    [0.5, 0.4, 0.1],   # Sunny: Walk=0.5, Shop=0.4, Clean=0.1
    [0.1, 0.3, 0.6],   # Rainy: Walk=0.1, Shop=0.3, Clean=0.6
])

# Sanity checks
assert np.allclose(pi.sum(), 1.0)
assert np.allclose(A.sum(axis=1), 1.0)
assert np.allclose(B.sum(axis=1), 1.0)
print(f"HMM: {N} hidden states, {M} observation symbols")
print(f"pi = {pi}")
print(f"A =\n{A}")
print(f"B =\n{B}")

### Generate sample data from the HMM

In [ ]:
def sample_hmm(pi, A, B, T):
    """Generate a sequence of length T from the HMM."""
    N, M = B.shape
    hidden = np.zeros(T, dtype=int)
    obs = np.zeros(T, dtype=int)
    
    hidden[0] = np.random.choice(N, p=pi)
    obs[0] = np.random.choice(M, p=B[hidden[0]])
    
    for t in range(1, T):
        hidden[t] = np.random.choice(N, p=A[hidden[t-1]])
        obs[t] = np.random.choice(M, p=B[hidden[t]])
    return hidden, obs

T = 10
true_hidden, obs_seq = sample_hmm(pi, A, B, T)
print("True hidden states:", [states[s] for s in true_hidden])
print("Observations:      ", [observations_vocab[o] for o in obs_seq])

## 2. Forward Algorithm

Computes $P(O \mid \lambda)$ — the likelihood of the observation sequence.

**Recurrence:**
$$\alpha_t(j) = \begin{cases} \pi_j \cdot b_j(o_1) & t = 1 \\ \left[\sum_{i=1}^{N} \alpha_{t-1}(i) \cdot a_{ij}\right] \cdot b_j(o_t) & t > 1 \end{cases}$$

**Result:** $P(O \mid \lambda) = \sum_{i=1}^N \alpha_T(i)$

Complexity: $O(N^2 T)$ vs $O(N^T)$ for brute-force enumeration.

In [ ]:
def forward(pi, A, B, obs):
    """Forward algorithm. Returns alpha table and log-likelihood.
    
    Args:
        pi: (N,) initial distribution
        A: (N,N) transition matrix
        B: (N,M) emission matrix
        obs: (T,) observation indices
    Returns:
        alpha: (T, N) forward probabilities
        log_prob: log P(obs | model)
    """
    T = len(obs)
    N = len(pi)
    alpha = np.zeros((T, N))
    
    # Initialization: alpha_1(j) = pi_j * b_j(o_1)
    alpha[0] = pi * B[:, obs[0]]
    
    # Induction
    for t in range(1, T):
        for j in range(N):
            alpha[t, j] = np.sum(alpha[t-1] * A[:, j]) * B[j, obs[t]]
    
    log_prob = np.log(np.sum(alpha[-1]))
    return alpha, log_prob

alpha, log_prob = forward(pi, A, B, obs_seq)
print(f"Log P(O|model) = {log_prob:.4f}")
print(f"P(O|model)     = {np.exp(log_prob):.6e}")
print(f"\nAlpha table (T={T}, N={N}):")
print(alpha)

### Scaled Forward (numerically stable)

For long sequences, $\alpha$ values underflow to 0. We normalize at each step:
$\hat{\alpha}_t(j) = \alpha_t(j) / c_t$ where $c_t = \sum_j \alpha_t(j)$.

Then $\log P(O|\lambda) = \sum_t \log c_t$.

In [ ]:
def forward_scaled(pi, A, B, obs):
    """Scaled forward algorithm — numerically stable."""
    T = len(obs)
    N = len(pi)
    alpha = np.zeros((T, N))
    scales = np.zeros(T)
    
    alpha[0] = pi * B[:, obs[0]]
    scales[0] = alpha[0].sum()
    alpha[0] /= scales[0]
    
    for t in range(1, T):
        for j in range(N):
            alpha[t, j] = np.sum(alpha[t-1] * A[:, j]) * B[j, obs[t]]
        scales[t] = alpha[t].sum()
        alpha[t] /= scales[t]
    
    log_prob = np.sum(np.log(scales))
    return alpha, scales, log_prob

alpha_s, scales, log_prob_s = forward_scaled(pi, A, B, obs_seq)
print(f"Scaled log P(O|model) = {log_prob_s:.4f}  (should match {log_prob:.4f})")

## 3. Backward Algorithm

Computes $\beta_t(i) = P(o_{t+1}, \ldots, o_T \mid z_t = i, \lambda)$.

$$\beta_t(i) = \begin{cases} 1 & t = T \\ \sum_{j=1}^N a_{ij} \cdot b_j(o_{t+1}) \cdot \beta_{t+1}(j) & t < T \end{cases}$$

In [ ]:
def backward(pi, A, B, obs):
    """Backward algorithm. Returns beta table."""
    T = len(obs)
    N = len(pi)
    beta = np.zeros((T, N))
    
    # Initialization: beta_T(i) = 1
    beta[-1] = 1.0
    
    # Induction (backwards)
    for t in range(T - 2, -1, -1):
        for i in range(N):
            beta[t, i] = np.sum(A[i, :] * B[:, obs[t+1]] * beta[t+1])
    
    return beta

def backward_scaled(pi, A, B, obs, scales):
    """Scaled backward algorithm (using forward scales)."""
    T = len(obs)
    N = len(pi)
    beta = np.zeros((T, N))
    beta[-1] = 1.0 / scales[-1]
    
    for t in range(T - 2, -1, -1):
        for i in range(N):
            beta[t, i] = np.sum(A[i, :] * B[:, obs[t+1]] * beta[t+1])
        beta[t] /= scales[t]
    
    return beta

beta = backward(pi, A, B, obs_seq)
# Verify: P(O|model) via backward = sum_i pi_i * b_i(o_1) * beta_1(i)
prob_backward = np.sum(pi * B[:, obs_seq[0]] * beta[0])
print(f"P(O|model) via forward:  {np.exp(log_prob):.6e}")
print(f"P(O|model) via backward: {prob_backward:.6e}")
print(f"Match: {np.allclose(np.exp(log_prob), prob_backward)}")

## 4. Viterbi Algorithm (Decoding)

Finds the single **most likely state sequence** via dynamic programming.

$$\delta_t(j) = \max_{z_1,\ldots,z_{t-1}} P(z_1,\ldots,z_{t-1}, z_t=j, o_1,\ldots,o_t \mid \lambda)$$

**Recurrence (in log-space):**
$$\delta_t(j) = \max_i \left[\delta_{t-1}(i) + \log a_{ij}\right] + \log b_j(o_t)$$

Keep a **backpointer** $\psi_t(j) = \arg\max_i [\delta_{t-1}(i) + \log a_{ij}]$ for traceback.

**Key distinction from Forward:** Forward *sums* over all paths; Viterbi takes the *max*.

In [ ]:
def viterbi(pi, A, B, obs):
    """Viterbi algorithm. Returns most likely state sequence and its log-probability.
    
    Works in log-space to avoid underflow.
    """
    T = len(obs)
    N = len(pi)
    
    log_pi = np.log(pi + 1e-300)
    log_A = np.log(A + 1e-300)
    log_B = np.log(B + 1e-300)
    
    # delta[t,j] = max log-prob of any path ending in state j at time t
    delta = np.zeros((T, N))
    psi = np.zeros((T, N), dtype=int)  # backpointers
    
    # Initialization
    delta[0] = log_pi + log_B[:, obs[0]]
    psi[0] = 0  # no predecessor at t=0
    
    # Recursion
    for t in range(1, T):
        for j in range(N):
            scores = delta[t-1] + log_A[:, j]  # (N,)
            psi[t, j] = np.argmax(scores)
            delta[t, j] = scores[psi[t, j]] + log_B[j, obs[t]]
    
    # Termination
    best_last = np.argmax(delta[-1])
    best_log_prob = delta[-1, best_last]
    
    # Backtrack
    path = np.zeros(T, dtype=int)
    path[-1] = best_last
    for t in range(T - 2, -1, -1):
        path[t] = psi[t + 1, path[t + 1]]
    
    return path, best_log_prob, delta, psi

viterbi_path, viterbi_log_prob, delta, psi = viterbi(pi, A, B, obs_seq)

print("Observations: ", [observations_vocab[o] for o in obs_seq])
print("True states:  ", [states[s] for s in true_hidden])
print("Viterbi path: ", [states[s] for s in viterbi_path])
print(f"Viterbi log-prob: {viterbi_log_prob:.4f}")
accuracy = np.mean(viterbi_path == true_hidden)
print(f"Accuracy vs true: {accuracy:.1%}")

## 5. Visualize: Trellis Diagram

In [ ]:
def plot_trellis(obs, states_list, obs_vocab, viterbi_path, delta, A):
    """Visualize the Viterbi trellis with the optimal path highlighted."""
    T = len(obs)
    N = len(states_list)
    
    fig, ax = plt.subplots(figsize=(max(12, T * 1.5), 4))
    
    # Draw all possible transitions (light gray)
    for t in range(T - 1):
        for i in range(N):
            for j in range(N):
                ax.plot([t, t+1], [i, j], 'lightgray', linewidth=0.5, zorder=1)
    
    # Highlight Viterbi path
    for t in range(T - 1):
        ax.plot([t, t+1], [viterbi_path[t], viterbi_path[t+1]], 
                'r-', linewidth=2.5, zorder=2)
    
    # Draw nodes
    for t in range(T):
        for i in range(N):
            color = 'red' if viterbi_path[t] == i else 'lightblue'
            ax.scatter(t, i, s=200, c=color, edgecolors='black', zorder=3)
            ax.text(t, i + 0.15, f"{delta[t,i]:.1f}", ha='center', fontsize=7)
    
    ax.set_xticks(range(T))
    ax.set_xticklabels([obs_vocab[o] for o in obs], fontsize=9)
    ax.set_yticks(range(N))
    ax.set_yticklabels(states_list, fontsize=10)
    ax.set_xlabel('Time / Observations')
    ax.set_ylabel('Hidden States')
    ax.set_title('Viterbi Trellis Diagram (numbers = log-delta, red = optimal path)')
    plt.tight_layout()
    plt.show()

plot_trellis(obs_seq, states, observations_vocab, viterbi_path, delta, A)

## 6. Visualize: Transition and Emission Probabilities

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Transition matrix
im0 = axes[0].imshow(A, cmap='Blues', vmin=0, vmax=1)
axes[0].set_xticks(range(N)); axes[0].set_xticklabels(states)
axes[0].set_yticks(range(N)); axes[0].set_yticklabels(states)
axes[0].set_xlabel('To state'); axes[0].set_ylabel('From state')
axes[0].set_title('Transition Matrix A')
for i in range(N):
    for j in range(N):
        axes[0].text(j, i, f"{A[i,j]:.2f}", ha='center', va='center', fontsize=12)
plt.colorbar(im0, ax=axes[0], fraction=0.046)

# Emission matrix
im1 = axes[1].imshow(B, cmap='Oranges', vmin=0, vmax=1)
axes[1].set_xticks(range(M)); axes[1].set_xticklabels(observations_vocab)
axes[1].set_yticks(range(N)); axes[1].set_yticklabels(states)
axes[1].set_xlabel('Observation'); axes[1].set_ylabel('State')
axes[1].set_title('Emission Matrix B')
for i in range(N):
    for k in range(M):
        axes[1].text(k, i, f"{B[i,k]:.2f}", ha='center', va='center', fontsize=12)
plt.colorbar(im1, ax=axes[1], fraction=0.046)

plt.tight_layout()
plt.show()

## 7. Baum-Welch Algorithm (EM for HMMs)

Learns HMM parameters from observations alone (unsupervised). This is Expectation-Maximization specialized for HMMs.

### E-step: Compute posterior quantities

**Gamma** (posterior state probability):
$$\gamma_t(i) = P(z_t = i \mid O, \lambda) = \frac{\alpha_t(i) \cdot \beta_t(i)}{P(O \mid \lambda)}$$

**Xi** (posterior transition probability):
$$\xi_t(i, j) = P(z_t=i, z_{t+1}=j \mid O, \lambda) = \frac{\alpha_t(i) \cdot a_{ij} \cdot b_j(o_{t+1}) \cdot \beta_{t+1}(j)}{P(O \mid \lambda)}$$

### M-step: Re-estimate parameters

$$\hat{\pi}_i = \gamma_1(i) \qquad \hat{a}_{ij} = \frac{\sum_{t=1}^{T-1} \xi_t(i,j)}{\sum_{t=1}^{T-1} \gamma_t(i)} \qquad \hat{b}_i(k) = \frac{\sum_{t: o_t=k} \gamma_t(i)}{\sum_{t=1}^{T} \gamma_t(i)}$$

In [ ]:
def baum_welch(obs, N, M, n_iter=50, tol=1e-6, verbose=True):
    """Baum-Welch (EM) algorithm for learning HMM parameters.
    
    Args:
        obs: (T,) observation sequence (integer indices)
        N: number of hidden states
        M: number of observation symbols
        n_iter: max EM iterations
        tol: convergence threshold on log-likelihood
    Returns:
        pi, A, B: learned parameters
        log_probs: list of log-likelihoods per iteration
    """
    T = len(obs)
    
    # Random initialization (Dirichlet-like)
    pi_est = np.random.dirichlet(np.ones(N))
    A_est = np.array([np.random.dirichlet(np.ones(N)) for _ in range(N)])
    B_est = np.array([np.random.dirichlet(np.ones(M)) for _ in range(N)])
    
    log_probs = []
    
    for iteration in range(n_iter):
        # --- E-step ---
        # Forward (scaled)
        alpha, scales, log_prob = forward_scaled(pi_est, A_est, B_est, obs)
        log_probs.append(log_prob)
        
        # Backward (scaled)
        beta = backward_scaled(pi_est, A_est, B_est, obs, scales)
        
        # Gamma: gamma[t,i] = P(z_t=i | O, lambda)
        gamma = alpha * beta
        gamma /= gamma.sum(axis=1, keepdims=True) + 1e-300
        
        # Xi: xi[t,i,j] = P(z_t=i, z_{t+1}=j | O, lambda)
        xi = np.zeros((T - 1, N, N))
        for t in range(T - 1):
            for i in range(N):
                for j in range(N):
                    xi[t, i, j] = (alpha[t, i] * A_est[i, j] * 
                                   B_est[j, obs[t+1]] * beta[t+1, j])
            xi[t] /= xi[t].sum() + 1e-300
        
        # --- M-step ---
        pi_est = gamma[0]
        
        for i in range(N):
            denom = gamma[:-1, i].sum() + 1e-300
            for j in range(N):
                A_est[i, j] = xi[:, i, j].sum() / denom
            
            denom_full = gamma[:, i].sum() + 1e-300
            for k in range(M):
                B_est[i, k] = gamma[obs == k, i].sum() / denom_full
        
        # Convergence check
        if verbose and (iteration + 1) % 10 == 0:
            print(f"  Iter {iteration+1}: log P(O|lambda) = {log_prob:.4f}")
        if len(log_probs) > 1 and abs(log_probs[-1] - log_probs[-2]) < tol:
            if verbose:
                print(f"  Converged at iteration {iteration+1}")
            break
    
    return pi_est, A_est, B_est, log_probs

In [ ]:
# Generate a longer sequence for learning
np.random.seed(42)
_, obs_long = sample_hmm(pi, A, B, T=200)
print(f"Training on {len(obs_long)} observations")

pi_learned, A_learned, B_learned, ll_history = baum_welch(
    obs_long, N=2, M=3, n_iter=100
)

In [ ]:
# Plot EM convergence
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(ll_history, 'b.-')
ax.set_xlabel('EM Iteration')
ax.set_ylabel('Log P(O | model)')
ax.set_title('Baum-Welch Convergence')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Compare learned vs true parameters
# Note: states may be permuted — HMMs have label-switching ambiguity
print("=== True Parameters ===")
print(f"pi = {pi}")
print(f"A =\n{A}")
print(f"B =\n{B}")
print()
print("=== Learned Parameters ===")
print(f"pi = {pi_learned}")
print(f"A =\n{A_learned}")
print(f"B =\n{B_learned}")
print()
print("Note: States may be permuted (label-switching symmetry).")
print("Compare row-by-row; if needed, swap rows to match.")

## 8. Posterior State Probabilities (Gamma)

Visualize $\gamma_t(i) = P(z_t = i \mid O, \lambda)$ — the smoothed posterior over hidden states.

In [ ]:
# Compute gamma for the short sequence using true parameters
alpha_full, scales_full, _ = forward_scaled(pi, A, B, obs_seq)
beta_full = backward_scaled(pi, A, B, obs_seq, scales_full)
gamma_full = alpha_full * beta_full
gamma_full /= gamma_full.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)

# Posterior probabilities
for i in range(N):
    axes[0].plot(gamma_full[:, i], 'o-', label=states[i], linewidth=2)
axes[0].set_ylabel('P(state | observations)')
axes[0].set_title('Posterior State Probabilities (gamma)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# True states vs Viterbi
axes[1].step(range(T), true_hidden, 'g-', where='mid', label='True', linewidth=2)
axes[1].step(range(T), viterbi_path, 'r--', where='mid', label='Viterbi', linewidth=2)
axes[1].set_yticks(range(N)); axes[1].set_yticklabels(states)
axes[1].set_xlabel('Time')
axes[1].set_title('True vs Viterbi Decoded States')
axes[1].legend()

plt.tight_layout()
plt.show()

## Interview Questions

### Q: "What are the three problems of HMMs?"

1. **Evaluation** (Forward): Given $\lambda$ and $O$, compute $P(O|\lambda)$. Used for model selection.
2. **Decoding** (Viterbi): Given $\lambda$ and $O$, find the most likely hidden state sequence. Used for tagging/labeling.
3. **Learning** (Baum-Welch): Given $O$, find $\lambda$ that maximizes $P(O|\lambda)$. Unsupervised parameter estimation.

### Q: "Viterbi vs Forward algorithm?"

| | Forward | Viterbi |
|---|---------|--------|
| Operation | **sum** over paths | **max** over paths |
| Output | $P(O \mid \lambda)$ | Best state sequence |
| Complexity | $O(N^2 T)$ | $O(N^2 T)$ |
| Needs backtracking? | No | Yes (backpointers) |

Both use the same DP structure — just replace $\sum$ with $\max$.

### Q: "HMM vs CRF?"

| | HMM | CRF |
|---|-----|-----|
| Type | **Generative** | **Discriminative** |
| Models | $P(X, Z)$ joint | $P(Z \mid X)$ conditional |
| Independence | Observations independent given states | Can use overlapping, arbitrary features of $X$ |
| Label bias | Suffers from it | Does not (global normalization) |
| Training | EM (Baum-Welch) or MLE | Gradient-based (convex objective) |

CRFs are generally preferred for sequence labeling (NER, POS tagging) because they can incorporate rich features and do not make the strong independence assumptions of HMMs.

### Q: "Applications of HMMs?"

- Speech recognition (before deep learning era)
- Part-of-speech tagging
- Gene finding in bioinformatics
- Financial regime detection (bull/bear markets)
- Gesture recognition